# Business Entity Resolution -- full pipeline on Kaggle (branch `features-address-name`)

**Before you run:** Settings -> Internet **ON**, Add-ons -> Secrets -> `GITHUB_TOKEN` attached to this notebook,
your private dataset added under *Input*. Then **Save Version -> Save & Run All (Commit)**.

Only `/kaggle/working/output` is saved as the notebook's output. Everything big lives in `/tmp/work` and is thrown away.

In [ ]:
# Cell 1 -- clone ONLY the branch, then remove the token from .git/config
import os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_REPO = "Payal-mak/amlc26-entity-resolution"      # change if the branch lives in your own fork
BRANCH = "features-address-name"
REPO_DIR = "/tmp/repo"                                   # /tmp: not part of the saved notebook output

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
clean_url = f"https://github.com/{GITHUB_REPO}.git"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True, encoding="utf-8", errors="replace")
    return r.returncode, (r.stdout + r.stderr).replace(token, "***")   # never print the token

shutil.rmtree(REPO_DIR, ignore_errors=True)
rc, out = git("clone", "-b", BRANCH, "--single-branch", clean_url.replace("https://", f"https://{token}@"), REPO_DIR)
print(out.strip()[-400:])
assert rc == 0, "clone failed -- is the GITHUB_TOKEN secret attached, and Internet ON?"

# immediately replace the remote URL with the token-free one
rc, out = git("-C", REPO_DIR, "remote", "set-url", "origin", clean_url)
assert rc == 0, out
assert token not in open(f"{REPO_DIR}/.git/config").read(), "token still in .git/config!"
print("remote:", git("-C", REPO_DIR, "remote", "-v")[1].strip().splitlines()[0])
print("commit:", git("-C", REPO_DIR, "log", "--oneline", "-1")[1].strip())
COMMIT = git("-C", REPO_DIR, "rev-parse", "HEAD")[1].strip()
print("full hash:", COMMIT)

In [ ]:
# Cell 2 -- install the pinned requirements (unrelated Kaggle-package conflict warnings are hidden)
import subprocess, sys
import importlib.metadata as md

REPO_DIR = globals().get("REPO_DIR", "/tmp/repo")
PIPELINE_DIR = f"{REPO_DIR}/student_resource/code/business_entity_resolution"

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PIPELINE_DIR}/requirements.txt"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
noise = ("dependency resolver", "which is incompatible", "requires ", "pip's")
lines = [l for l in (r.stdout + r.stderr).splitlines() if l.strip() and not any(n in l for n in noise)]
print("\n".join(lines[-15:]) or "(no pip output)")
print("pip exit code:", r.returncode)
assert r.returncode == 0, "pip install failed"
for pkg in ["duckdb", "pandas", "pyarrow", "lightgbm", "rapidfuzz", "indic-transliteration", "scikit-learn", "psutil"]:
    print(f"  {pkg:24s}{md.version(pkg)}")

In [ ]:
# Cell 3 -- find the dataset folder (the one that contains train/train_source1.tsv and test/test_source1.tsv)
import os

INPUT_ROOT = globals().get("INPUT_ROOT", "/kaggle/input")
NEEDED = ["train/train_source1.tsv", "train/train_source2.tsv", "train/train_source3.tsv", "train/train_ground_truth.tsv",
          "test/test_source1.tsv", "test/test_source2.tsv", "test/test_source3.tsv"]

def find_dataset_dirs(root, max_depth=6):
    hits = []
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath[len(root):].count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if all(os.path.isfile(os.path.join(dirpath, f)) for f in NEEDED[:1] + NEEDED[4:5]):
            hits.append(dirpath)
            dirnames[:] = []          # don't descend into a dataset folder
    return hits

hits = find_dataset_dirs(INPUT_ROOT)
if not hits:
    print("Nothing found under", INPUT_ROOT, "-- top level is:", os.listdir(INPUT_ROOT) if os.path.isdir(INPUT_ROOT) else "missing")
    raise SystemExit("Add your private dataset under Input (it must contain train/ and test/ folders).")
DATA_DIR = hits[0]
if len(hits) > 1:
    print("WARNING: several matches, using the first:", hits)
missing = [f for f in NEEDED if not os.path.isfile(os.path.join(DATA_DIR, f))]
assert not missing, f"dataset folder {DATA_DIR} is missing {missing}"
print("DATA_DIR =", DATA_DIR)
for f in NEEDED:
    print(f"  {f:34s}{os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6:9.1f} MB")

In [ ]:
# Cell 4 -- run the whole pipeline (train + test), printing time and peak RAM for every stage
import os, subprocess, sys, time

REPO_DIR = globals().get("REPO_DIR", "/tmp/repo")
PIPELINE_DIR = f"{REPO_DIR}/student_resource/code/business_entity_resolution"
WORK_DIR = "/tmp/work"
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/output")
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- one run (default) or two runs (only if one run would not fit the 12-hour limit) -------------------
#   "single": every stage in this notebook run.
#   "A":      normalize ... val_train, then save a small "carry" folder (models + phase-2 files) to /kaggle/working/carry.
#   "B":      restore that carry folder from an attached dataset (CARRY_INPUT), then test_features -> predict -> write.
MODE = "single"
CARRY_INPUT = "/kaggle/input/aml-carry"            # only used by MODE = "B"

STAGES = {
    "single": ["all"],
    "A": ["normalize", "val_slice", "val_blocking", "val_features", "val_train"],
    "B": ["test_features", "predict", "write"],
}[MODE]

# Same AML_* names src/config.py reads.
env = dict(
    os.environ,
    AML_DATA_DIR=DATA_DIR,
    AML_WORK_DIR=WORK_DIR,
    AML_OUTPUT_DIR=OUTPUT_DIR,
    AML_DUCKDB_MEMORY_LIMIT="14GB",     # NOT 20GB: pandas frames live in the same 30GB; DuckDB spills to disk past its limit
    AML_DUCKDB_THREADS="4",
    AML_LGBM_THREADS="4",
    PYTHONUNBUFFERED="1",
)
# --skip-full-train: predict the test set with the model fitted on the validation slice (val_train). The default
# path fits on ALL train pairs (~100M rows x ~46 features), which cannot fit in 30GB (README.md, "Scale").
# DISABLE_TFIDF: the char-3-gram TF-IDF block has never been run at full-country scale (README/PROJECT_LOG); it is
# left OFF here, as in the team's own plan. Set False only if you have hours to spare and want to watch it.
DISABLE_TFIDF = True
EXTRA_ARGS = ["--skip-full-train"] + (["--disable-tfidf-block"] if DISABLE_TFIDF else [])

if MODE == "B":
    import shutil
    for rel in ["parquet/phase2", "models", "parquet/pipeline/train_report.json", "parquet/phase4/report.json"]:
        src, dst = os.path.join(CARRY_INPUT, rel), os.path.join(WORK_DIR, rel)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif os.path.isfile(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
        else:
            print("WARNING: carry file missing:", rel)

t_all = time.time()
log = open(os.path.join(OUTPUT_DIR, "pipeline_log.txt"), "a", encoding="utf-8")
for stage in STAGES:
    cmd = [sys.executable, "-u", "-m", "scripts.run_pipeline", "--stage", stage, *EXTRA_ARGS]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=PIPELINE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:              # run_pipeline prints "=== stage: X done (Ns, peak_rss=NMB) ===" for every stage
        print(line, end="")
        log.write(line)
    p.wait()
    log.flush()
    assert p.returncode == 0, f"stage {stage} failed with exit code {p.returncode} -- see the log above"
log.close()
print(f"\nTOTAL wall time this cell: {(time.time() - t_all) / 3600:.2f} h")

if MODE == "A":
    import shutil
    carry = "/kaggle/working/carry"
    for rel in ["parquet/phase2", "models", "parquet/pipeline/train_report.json", "parquet/phase4/report.json"]:
        src, dst = os.path.join(WORK_DIR, rel), os.path.join(carry, rel)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif os.path.isfile(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
    print("carry folder written to", carry, "-- after this run: Output tab -> New Dataset, name it aml-carry")

In [ ]:
# Cell 5 -- check the two submission files
import os, subprocess, sys

REPO_DIR = globals().get("REPO_DIR", "/tmp/repo")
DATA_DIR = globals()["DATA_DIR"]
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/output")
M, C = f"{OUTPUT_DIR}/matching_results.tsv", f"{OUTPUT_DIR}/candidate_pairs.tsv"

r = subprocess.run([sys.executable, f"{REPO_DIR}/student_resource/utils/validate_submission.py",
                    "--matching", M, "--candidate", C, "--test-dir", f"{DATA_DIR}/test", "--check-ids"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print((r.stdout + r.stderr)[-2500:])
print("VALIDATOR:", "PASS" if r.returncode == 0 else "FAIL", f"(exit code {r.returncode})")

n_test_s1 = sum(1 for _ in open(f"{DATA_DIR}/test/test_source1.tsv", encoding="utf-8")) - 1
rows = empty_matches = empty_candidates = n_matched_ids = not_in_candidates = 0
examples = []
with open(M, encoding="utf-8") as fm, open(C, encoding="utf-8") as fc:
    assert fm.readline().rstrip("\n").split("\t") == ["source1_entity_id", "matched_entity_ids"], "bad matching header"
    assert fc.readline().rstrip("\n").split("\t") == ["source1_entity_id", "candidate_entity_ids"], "bad candidate header"
    for lm, lc in zip(fm, fc):                        # both files are written in the same sorted S1 order
        s1m, _, matched = lm.rstrip("\n").partition("\t")
        s1c, _, cands = lc.rstrip("\n").partition("\t")
        assert s1m == s1c, f"files are not in the same S1 order: {s1m} vs {s1c}"
        rows += 1
        if not matched:
            empty_matches += 1
        if not cands:
            empty_candidates += 1
        if matched:
            cset = set(cands.split(","))
            for m in matched.split(","):
                n_matched_ids += 1
                if m not in cset:
                    not_in_candidates += 1
                    if len(examples) < 5:
                        examples.append((s1m, m))
print(f"test S1 rows expected: {n_test_s1:,}   matching rows: {rows:,}   candidate rows: {rows:,}")
print(f"S1 with EMPTY matches: {empty_matches:,} ({empty_matches / max(rows, 1):.1%})   S1 with empty candidates: {empty_candidates:,}")
print(f"matched ids: {n_matched_ids:,}   NOT in candidate_pairs.tsv: {not_in_candidates:,}", "-> OK" if not_in_candidates == 0 else f"-> PROBLEM, e.g. {examples}")
for f in (M, C):
    print(f"  {os.path.basename(f)}: {os.path.getsize(f) / 1e6:.0f} MB")
ok = r.returncode == 0 and rows == n_test_s1 and not_in_candidates == 0
print("\nOVERALL:", "PASS" if ok else "FAIL")

In [ ]:
# Cell 6 -- keep the validation report with the outputs (it survives Save & Run All)
import json, os

WORK_DIR = globals().get("WORK_DIR", "/tmp/work")
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/output")
COMMIT = globals().get("COMMIT", "unknown")
P4 = f"{WORK_DIR}/parquet/phase4"

def load(path):
    return json.load(open(path, encoding="utf-8")) if os.path.isfile(path) else {}

val = load(f"{P4}/report.json")                                   # written by val_train (validation-slice OOF)
train = load(f"{WORK_DIR}/parquet/pipeline/train_report.json")    # written by train
singleton_fp = n_singletons = None
try:
    import pandas as pd
    gt = pd.read_parquet(f"{WORK_DIR}/parquet/val_slice/ground_truth_in_slice.parquet")
    s1 = pd.read_parquet(f"{WORK_DIR}/parquet/val_slice/source1.parquet", columns=["entity_id", "is_eval"])
    ev = set(s1.loc[s1["is_eval"], "entity_id"])
    truth = {a: (b if isinstance(b, str) and b.strip() else "") for a, b in zip(gt["source1_entity_id"], gt["matched_entity_ids"]) if a in ev}
    pred = pd.read_parquet(f"{P4}/val_predictions.parquet")
    pred = {a: (b if isinstance(b, str) and b.strip() else "") for a, b in zip(pred["source1_entity_id"], pred["matched_entity_ids"])}
    singles = [a for a, t in truth.items() if not t]
    n_singletons, singleton_fp = len(singles), sum(1 for a in singles if pred.get(a))
except Exception as e:                                              # the report must still be written
    print("singleton FP count unavailable:", e)

by_group = val.get("by_group", {})
summary = {
    "commit": COMMIT,
    "branch": "features-address-name",
    "validation_slice_oof": {
        "macro_f0.5": val.get("macro_f_beta"),
        "micro_precision": val.get("micro_precision"),
        "micro_recall": val.get("micro_recall"),
        "singleton_accuracy": val.get("singleton_accuracy"),
        "singleton_false_positives": singleton_fp,
        "n_singletons": n_singletons,
        "policy": val.get("policy"),
        "India_f0.5": (by_group.get("India") or {}).get("f_beta"),
        "US_f0.5": (by_group.get("US") or {}).get("f_beta"),
        "n_entities": val.get("n_entities"),
    },
    "train_stage": {k: train.get(k) for k in ("n_train_rows", "n_positives", "macro_f_beta", "policy")},
    "full_val_report": val,
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f"{OUTPUT_DIR}/report.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str, ensure_ascii=False)
print(json.dumps({k: v for k, v in summary.items() if k != "full_val_report"}, indent=2, default=str))
total_mb = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk("/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_DIR) for f in fs) / 1e6
print(f"\nsize of the saved output folder: {total_mb:.0f} MB (limit 20,000 MB)")